# A Simple Iterative Proof Development System

This notebook runs a loop that calls an LLM twice each loop iteration. In each call, the LLM is given a role as an *agent* with a specific system prompt. A **prover agent** tries to prove or disprove a conjecture, and a **referee agent** checks the prover's argument for errors. The loop stops when the referee accepts the argument or when the maximum number of rounds has been reached.

**If this is your first time using a Jupyter notebook:**
To run a cell, bring it to focus (e.g. by clicking on it so that it highlights) and then press **Shift + Enter**. Run each cell in order, as some cells further down depend on the actions of earlier cells.


## **LLM Set-Up**

In [ ]:
%pip install -q openai

Set the provider's base URL and then paste your API key when prompted. The key is hidden and is not saved in the notebook. (For Gemini, use `https://generativelanguage.googleapis.com/v1beta/openai/` as the base URL.)

In [ ]:
from getpass import getpass
from openai import OpenAI

base_url = "PASTE_BASE_URL_HERE"
api_key = getpass("API key: ")

client = OpenAI(base_url=base_url, api_key=api_key)

List the available models, then copy one name into `MODEL` below.

In [ ]:
for model in client.models.list():
    print(model.id)

In [ ]:
MODEL = "PASTE_MODEL_HERE"

def ask(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt}, # The system prompt tells the LLM how it should behave or act
            {"role": "user", "content": user_prompt}, # The user prompt is the query that a user asks the LLM
        ],
    )
    return response.choices[0].message.content

## **Agent Prompts**

The conjecture to prove or disprove. (You can try experimenting with the content of the conjecture to see a model's capabilities.)

In [ ]:
CONJECTURE = "For every integer n >= 1, the number n^2 + n + 41 is prime."

The prover's instructions:

In [ ]:
PROVER_PROMPT = """You are a careful mathematician. Given a conjecture, either prove it or disprove it.
To disprove it, give an explicit counterexample and verify that it is a valid counterexample.
If you are shown a previous attempt and referee feedback, use the feedback to either revise or improve your work.
Write a complete, rigorous argument."""

The referee's instructions (the last line of the referee's reply acts as the signal to stop the loop):

In [ ]:
REFEREE_PROMPT = """You are a careful referee. Check the proposed argument step by step for errors, gaps, or insufficiently justified claims.
List every problem you find.
End your reply with exactly one line: VERDICT: ACCEPT if the argument is correct and complete, or VERDICT: REJECT otherwise."""

## **Proof Iteration**

The main loop. In each round the prover writes an argument, and the referee reviews it. If the referee accepts, the `accepted` flag is set and the loop ends early.

In [ ]:
MAX_ROUNDS = 5

proof, review = "", ""
accepted = False

for i in range(1, MAX_ROUNDS + 1):
    request = f"Conjecture: {CONJECTURE}"
    if review:
        request += f"\n\nYour previous attempt:\n{proof}\n\nReferee feedback:\n{review}"

    proof = ask(PROVER_PROMPT, request)
    review = ask(REFEREE_PROMPT, f"Conjecture: {CONJECTURE}\n\nProposed argument:\n{proof}")

    print(f"===== Round {i} =====\n\nPROVER:\n{proof}\n\nREFEREE:\n{review}\n")

    if "VERDICT: ACCEPT" in review:
        accepted = True
        break

print("Accepted by the referee." if accepted else f"Not accepted after {MAX_ROUNDS} rounds.")
print("\nFinal argument:\n")
print(proof)